# Metrics Cleaning

This notebook cleans up the created error metrics by removing the extreme outliers which are present in each hierarchy. This isn't bad because we are focusing only on the relative performance of different hierarchies, not their absolute errors, and outliers would only make the analysis harder.

In [ ]:
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root

import python.utils as utils
import python.plots as plots
from python.time_series import BASE_FORECASTS, RECONCILED_FORECASTS, TimeSeriesType
from python.aggregation import MONTHS, FORTNIGHTS, WEEKS, WEEKDAYS, WEEKENDS, DAYS
from python.temporal_hierarchy import TemporalHierarchy, MinimalTemporalHierarchy, SemanticTemporalHierarchy
from python.error_metrics import ErrorMetrics, MergedErrorMetrics, MetricsIndex, LoadedErrorMetrics, ErrorDifference, FilteredErrorMetrics, ClonedErrorMetrics
from python.error_metrics import MetricsIndex as MI # Alias
from python.forecast_reconciliation import ForecastReconciliation

First we import the raw error metrics

In [ ]:
em = LoadedErrorMetrics('metrics/base_hier_errors.csv')

In [ ]:
em.get_error_stats(MI(forecast_type=BASE_FORECASTS, error_metric='mase'))

Base forecast errors are identical for each hierarchy, of course, so for later analysis, most rows are omitted.

In [ ]:
em = MergedErrorMetrics([
    FilteredErrorMetrics(em, MI(forecast_type=RECONCILED_FORECASTS)),
    FilteredErrorMetrics(em, MI( forecast_type=BASE_FORECASTS, hierarchy='Minimal')),
])

In [ ]:
em.save('metrics/base_hier_errors.csv')

Identifying outliers by their MASE score being above the 90th percentile.

In [ ]:
row_idx = MI(forecast_type=BASE_FORECASTS, granularity=[DAYS], error_metric='mase')
em.get_error_stats(row_idx)


In [ ]:
q90 = em.get_error_stats(row_idx)['q90'].iloc[0]

In [ ]:
err = em.get_error(row_idx).T
big_errors_table = err[err > q90].dropna().sort_values(by=err.columns[0], ascending=False)
big_errors_cols = big_errors_table.index.tolist()
big_errors_table

In [ ]:
em.drop_cols(big_errors_cols)

During the research some other extreme outliers were also noticed, which really messed up the visualisation, these ones are also removed here.

In [ ]:
row = em.get_error(MI(forecast_type=BASE_FORECASTS, error_metric='nrmse'))

col = row[(row - 0.01849427).abs() < 1e-7].dropna(axis=1, how='all').columns[0]
col2 = row[(row - 0.5305789).abs() < 1e-6].dropna(axis=1, how='all').columns[0]

em.drop_cols([col, col2])

In [ ]:
row = em.get_error(MI(forecast_type=RECONCILED_FORECASTS, hierarchy='Weeks+Biweeks', error_metric='nrmse', granularity=DAYS))

col3 = row[(row - 593.214660).abs() < 1e-7].dropna(axis=1, how='all').columns[0]

em.drop_cols([col3])

We can see by the stats that the errors have more or less been cleaned up from outliers 

In [ ]:
em.get_error_stats(MI(forecast_type=BASE_FORECASTS))

## Saving the cleaned metrics

In [ ]:
em.save('metrics/base_hier_errors_clean.csv')

## Cleaning the inconsistency errors
Dropping the same columns we have dropped from the inconsistency error table too.

In [ ]:
import pandas as pd

cols = big_errors_cols + [col, col2, col3]
inc = pd.read_csv('metrics/base_hier_inconsistency.csv', index_col=0)
inc = inc.drop(columns=cols)
inc.to_csv('metrics/base_hier_inconsistency_cleaned.csv')